In [1]:
import os
os.chdir("/home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project")
import pandas as pd
from utils.download_data import download_snotel_data
from metloom.pointdata import CDECPointData, SnotelPointData
from metloom.variables import CdecStationVariables, SnotelVariables
import geopandas as gpd

In [7]:
basin_shapes = {"tuolumne": "/storage/dlhogan/ess_project_data/domain_Tuolumne_River/shapefiles/catchment/Toulumne_HRUs_GRUs.shp",
                "east" : "/scratch/dlhogan/ess-project-data/domain_East_River_lumped/shapefiles/catchment/East_River_lumped_HRUs_GRUs.shp"}
tuolumne_obj = gpd.read_file(basin_shapes["tuolumne"])
east_obj = gpd.read_file(basin_shapes["east"])

CDEC_VARS = [
    CdecStationVariables.SWE,
    CdecStationVariables.TEMP,
]
SNOTEL_VARS = [
    SnotelVariables.SWE,
    SnotelVariables.TEMP,
]

In [26]:
points = CDECPointData.points_from_geometry(tuolumne_obj, CDEC_VARS, snow_courses=False)
tuolumne_df = pd.DataFrame(points)
points = SnotelPointData.points_from_geometry(east_obj, SNOTEL_VARS, snow_courses=False)
east_df = pd.DataFrame(points)
# add row for schofield pass
schofield = SnotelPointData("737:CO:SNTL", "SCHOFIELD PASS")
schofield_df = pd.DataFrame([schofield])
east_df = pd.concat([east_df, schofield_df], ignore_index=True)

basin_shapes = {"tuolumne": tuolumne_df, "east": east_df}

In [13]:
for i in range(len(tuolumne_df)):
    site_name = basin_shapes["tuolumne"][0].iloc[i].name.replace(" ", "_")
    site_id = tuolumne_df[0].iloc[i].id
    print("Downloading data for " + site_name + " with site ID " + site_id)
    try: 
        tuolumne_sntl = download_snotel_data(
                        start_date="1980-12-01",
                        end_date="2024-12-31",
                        variables='default',
                        site_id=site_id,
                        site_name=site_name,
                        needs_cdec=True
                        )
    except Exception as e:
        print(f"Failed to download data for {site_name} with site ID {site_id}: {e}")
        continue
    if tuolumne_sntl is not None:
        tuolumne_sntl.to_csv(f"/scratch/dlhogan/ess-project-data/meteorological-data/SNOTEL/{site_id}_{site_name}_cdec_obs.csv")
    print(f"Data for {site_name} with site ID {site_id} downloaded and saved.")

Failed to download data for TUOLUMNE_MEADOWS with site ID TUM: type object 'CdecStationVariables' has no attribute 'SOILMOISTURE2IN'
Failed to download data for DANA_MEADOWS with site ID DAN: type object 'CdecStationVariables' has no attribute 'SOILMOISTURE2IN'
Failed to download data for SLIDE_CANYON with site ID SLI: type object 'CdecStationVariables' has no attribute 'SOILMOISTURE2IN'
Failed to download data for TUOLUMNE_R_AT_THE_GRAND_CYN_OF_TUOLUMNE with site ID TGC: type object 'CdecStationVariables' has no attribute 'SOILMOISTURE2IN'
Failed to download data for GAYLOR_PIT with site ID GYP: type object 'CdecStationVariables' has no attribute 'SOILMOISTURE2IN'


In [ ]:
for i in range(len(basin_shapes["east"])):
    site_name = basin_shapes["east"][0].iloc[i].name.replace(" ", "_")
    site_id = east_df[0].iloc[i].id
    print("Downloading data for " + site_name + " with site ID " + site_id)
    try: 
        east_sntl = download_snotel_data(
                        start_date="1980-10-01",
                        end_date="2025-09-30",
                        variables='default',
                        site_id=site_id,
                        site_name=site_name,
                        needs_cdec=False
                        )
    except Exception as e:
        print(f"Failed to download data for {site_name} with site ID {site_id}: {e}")
        continue
    if east_sntl is not None:
        east_sntl.to_csv(f"/scratch/dlhogan/ess-project-data/meteorological-data/SNOTEL/{site_id.split(':')[0]}_{site_name}_sntl_obs.csv")
    print(f"Data for {site_name} with site ID {site_id} downloaded and saved.")

No Relative Humidity found for SCHOFIELD_PASS
No SOIL MOISTURE -4IN found for SCHOFIELD_PASS


Data for SCHOFIELD_PASS with site ID 737:CO:SNTL downloaded and saved.
